In [1]:
import os
import psutil
from cochem_bench.literature import xml_performance, XMLMemoryLimitExceeded

class DummyXMLStream:
    def __init__(self, num_nodes):
        self.num_nodes = num_nodes
        self.current = 0
        self.header = b'<root>\n'
        self.footer = b'</root>\n'
        self.node_template = b'<publication><id>%010d</id><text>' + (b'x' * 2000) + b'</text></publication>\n'
        self.state = 0
        self.buffer = b''

    def read(self, size=-1):
        if size == -1: size = 8192
        while len(self.buffer) < size and self.state < 3:
            if self.state == 0:
                self.buffer += self.header
                self.state = 1
            elif self.state == 1:
                chunk_nodes = min(100, self.num_nodes - self.current)
                if chunk_nodes > 0:
                    for i in range(chunk_nodes):
                        self.buffer += self.node_template % (self.current + i)
                    self.current += chunk_nodes
                else:
                    self.state = 2
            elif self.state == 2:
                self.buffer += self.footer
                self.state = 3
        
        ret = self.buffer[:size]
        self.buffer = self.buffer[size:]
        return ret


In [2]:
import time

print("Starting memory test with 2GB payload (1,000,000 nodes)...")
process = psutil.Process(os.getpid())
initial_memory = process.memory_info().rss

stream = DummyXMLStream(1000000)

start_time = time.time()
try:
    xml_performance(stream)
    print("Parsing completed successfully!")
except XMLMemoryLimitExceeded as e:
    print(f"FAILED: {e}")
except Exception as e:
    print(f"FAILED with unexpected error: {e}")

final_memory = process.memory_info().rss
peak_usage_mb = (final_memory - initial_memory) / (1024 * 1024)

print(f"Time taken: {time.time() - start_time:.2f} seconds")
print(f"Memory increase: {peak_usage_mb:.2f} MB")

# Validation: Assert RAM remains stable at <200 MB
assert peak_usage_mb < 200, f"Memory usage too high: {peak_usage_mb:.2f} MB"
print("Validation passed: RAM remained < 200 MB")


Starting memory test with 2GB payload (1,000,000 nodes)...


Parsing completed successfully!
Time taken: 15.47 seconds
Memory increase: 4.21 MB
Validation passed: RAM remained < 200 MB
